# 05 · Models

All models share one interface: `fit(visible, history)` then `predict(k, T)`.

| Code | Model | Kind | File |
|---|---|---|---|
| — | Mean IV of the day | floor | `src/models/dummy.py` |
| B0 | Linear interpolation of total variance in (k, T) | per day | `src/models/interp.py` |
| B1 | **SVI per expiry** (Gatheral–Jacquier constraints) — main benchmark | per day | `src/models/svi.py` |
| B2 | SSVI (one surface per day, butterfly penalties, θ increasing in T) | per day | `src/models/ssvi.py` |
| M1a | XGBoost predicting IV directly | learned, monthly | `src/models/xgb.py` |
| M1b | XGBoost predicting the error of linear interpolation | learned, monthly | `src/models/xgb.py` |
| M2 | XGBoost predicting the error of SVI | learned, monthly | `src/models/xgb.py` |
| M3 | Gaussian process (trend + Matern 5/2), per day | per day | `src/models/gp.py` |
| M4 | Neural net on an SSVI prior with no-arbitrage penalty (Deep Smoothing style) | per day | `src/models/nn.py` |
| M4′ | Same net without the penalty (ablation) | per day | `src/models/nn.py` |

**Learned models** are retrained at the start of every test month on all earlier days only
(walk-forward). Their training rows come from past days masked exactly like test days
(`src/ml_features.py`), so they learn to fill gaps from visible quotes, never from hidden ones.

**Full run:** `python -m src.run_all` from the project folder runs every model on every test day
(March 2020 – December 2022) and every mask, then the sparsity sweep. It is resumable and takes
a few hours; results land in `results/<model>/`. This notebook only demonstrates the models on
single days and reads the saved results.

In [ ]:
import sys
sys.path.append('..')
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160)

In [ ]:
from src.masking import random_mask, wing_mask, expiry_mask
from src.metrics import rmse
from src.arbitrage import check_surface
from src.models.dummy import MeanModel
from src.models.interp import LinearInterpModel
from src.models.svi import SVIModel
from src.models.ssvi import SSVIModel
from src.models.xgb import XGBModel, XGBLinearModel, XGBSVIModel
from src.models.gp import GPModel
from src.models.nn import NNModel, NNNoPenaltyModel
from src.evaluate import history_for

df = pd.read_parquet('../data/processed/options_features.parquet')
df.shape

## Optional: run everything from here
Leave `RUN_ALL = False` unless results are missing. It calls the same script as the command line.

In [ ]:
RUN_ALL = False
if RUN_ALL:
    from src.run_all import main
    main()

## All models on one crisis day
16 March 2020 — the day SPY fell 11%. Learned models are trained on history before March 2020
(January–February only), exactly as in the walk-forward test.

In [ ]:
DAY = pd.Timestamp('2020-03-16')
day = df[df['quote_date'] == DAY]
hist = history_for(df, DAY)

PER_DAY = [MeanModel, LinearInterpModel, SVIModel, SSVIModel, GPModel, NNModel, NNNoPenaltyModel]
LEARNED = [XGBModel, XGBLinearModel, XGBSVIModel]
states = {M.name: M.train(hist) for M in LEARNED}       # trained once, used for both masks

def fit_all(visible):
    models = {M.name: M().fit(visible) for M in PER_DAY}
    models.update({M.name: M(state=states[M.name]).fit(visible) for M in LEARNED})
    return models

rows, fitted = [], {}
for mask_name, fn in {'random': lambda d: random_mask(d, seed=0), 'wing': wing_mask,
                      'expiry': lambda d: expiry_mask(d, seed=0)}.items():
    m = fn(day)
    vis, hid = day[~m], day[m]
    fitted[mask_name] = (vis, hid, fit_all(vis))
    for name, model in fitted[mask_name][2].items():
        pred = model.predict(hid['k'].to_numpy(), hid['T'].to_numpy())
        rows.append({'model': name, 'mask': mask_name, 'rmse': rmse(hid['iv'], pred),
                     **check_surface(model, day)})
one_day = pd.DataFrame(rows)
one_day.pivot_table(index='model', columns='mask', values='rmse').round(2).sort_values('wing')

In [ ]:
def plot_slices(mask_name, models_to_show, n_slices=3):
    vis, hid, models = fitted[mask_name]
    Ts = np.sort(day['T'].unique())
    picks = Ts[np.linspace(0, len(Ts) - 1, n_slices + 2).astype(int)[1:-1]]
    fig, axes = plt.subplots(1, n_slices, figsize=(16, 4.5))
    for ax, T in zip(axes, picks):
        v, h = vis[np.isclose(vis['T'], T)], hid[np.isclose(hid['T'], T)]
        kk = np.linspace(day.loc[np.isclose(day['T'], T), 'k'].min(),
                         day.loc[np.isclose(day['T'], T), 'k'].max(), 200)
        ax.scatter(v['k'], v['iv'], s=8, c='black', label='visible')
        ax.scatter(h['k'], h['iv'], s=14, facecolors='none', edgecolors='red', label='hidden')
        for name in models_to_show:
            ax.plot(kk, models[name].predict(kk, np.full_like(kk, T)), lw=1.3, label=name)
        ax.set_title(f'{mask_name} mask, T = {T:.2f}'); ax.set_xlabel('k')
    axes[0].set_ylabel('implied volatility'); axes[0].legend(fontsize=8)
    plt.tight_layout(); plt.show()

plot_slices('wing', ['svi', 'ssvi', 'xgb_svi', 'gp', 'nn'])

In [ ]:
plot_slices('random', ['linear', 'svi', 'xgb_svi', 'nn'])

## Arbitrage on that day
Share of grid points violating the butterfly condition (g(k) < 0) and the calendar condition
(total variance falling with maturity). See `src/arbitrage.py`.

In [ ]:
one_day[one_day['mask'] == 'random'].set_index('model')[
    ['rmse', 'butterfly_rate', 'calendar_rate']].round(3).sort_values('rmse')

## What does XGBoost look at?
Feature importance (gain) of the SVI-error model trained on all history before December 2022.

In [ ]:
booster = XGBSVIModel.train(history_for(df, '2022-12-01'))
imp = pd.Series(booster.get_booster().get_score(importance_type='gain')).sort_values()
imp.plot.barh(figsize=(7, 5), title='XGBoost (SVI error) — feature importance, gain')
plt.show()

## Gaussian-process uncertainty
The GP also gives a confidence band. Wider bands where quotes were hidden = the model knows
it is extrapolating.

In [ ]:
vis, hid, models = fitted['wing']
T = np.sort(day['T'].unique())[len(day['T'].unique()) // 3]
kk = np.linspace(day.loc[np.isclose(day['T'], T), 'k'].min() - 0.05,
                 day.loc[np.isclose(day['T'], T), 'k'].max() + 0.05, 200)
mean, sd = models['gp'].predict(kk, np.full_like(kk, T), return_std=True)
v, h = vis[np.isclose(vis['T'], T)], hid[np.isclose(hid['T'], T)]
plt.figure(figsize=(8, 4.5))
plt.fill_between(kk, mean - 2 * sd, mean + 2 * sd, alpha=0.25, label='GP ±2 sd')
plt.plot(kk, mean, label='GP mean')
plt.scatter(v['k'], v['iv'], s=8, c='black', label='visible')
plt.scatter(h['k'], h['iv'], s=14, facecolors='none', edgecolors='red', label='hidden (wings)')
plt.title(f'Gaussian process, wing mask, T = {T:.2f}'); plt.xlabel('k'); plt.ylabel('iv')
plt.legend(); plt.show()

## Full-run summary
Mean daily RMSE (vol points) over all test days, read from `results/`. The full analysis
is in `06_Analysis.ipynb`.

In [ ]:
import os
ORDER = ['mean', 'linear', 'svi', 'ssvi', 'xgb', 'xgb_lin', 'xgb_svi', 'gp', 'nn', 'nn_nopen']
summ = {m: pd.read_csv(f'../results/{m}/summary_by_mask.csv', index_col=0)
        for m in ORDER if os.path.exists(f'../results/{m}/summary_by_mask.csv')}
pd.concat({m: s['rmse'] for m, s in summ.items()}, axis=1).T.round(2)